In [1]:
import sys
import os

current_file_path = os.path.abspath(os.getcwd()) # Funciona solo para JupyterNotebooks

# Obtener la ruta del directorio raíz del proyecto
project_root = os.path.abspath(os.path.join(os.path.dirname(current_file_path)))

# Añadir el directorio raíz del proyecto al sys.path
sys.path.append(project_root)

from config import INPUTS_FOLDER
import os

statement_path = os.path.join(INPUTS_FOLDER, 'test_files', 'BBVA', 'BBVA_credit_statement.pdf')
statement_path

'c:\\Proyectos\\Aether\\Aether_v1\\documents\\inputs\\test_files\\BBVA\\BBVA_credit_statement.pdf'

In [2]:
from document_reader import PDFReader

reader = PDFReader(statement_path)
document_height = reader.get_height()
extracted_words = reader.extract_words_from_pdf()

extracted_words

,page,text,x0,top,x1,bottom
0,1,Estado,505.170000,4.087000,541.265995,16.087000
1,1,de,544.025896,4.087000,557.189812,16.087000
2,1,Cuenta,559.937761,4.087000,597.629720,16.087000
3,1,Tarjeta,492.793794,17.107997,524.847394,27.407997
4,1,Platinum,527.710794,17.107997,567.788094,27.407997
...,...,...,...,...,...,...
2696,8,5552262663,321.099000,776.832650,365.579000,784.832650
2697,8,LARGA,367.803000,776.832650,394.923000,784.832650
2698,8,DISTANCIA,397.147000,776.832650,439.819000,784.832650
2699,8,SIN,442.043000,776.832650,455.379000,784.832650


In [3]:
from bank_detector import DefaultBankDetector

bank_detector = DefaultBankDetector(extracted_words, document_height)

bank = bank_detector.detect_bank()
statement_type = bank_detector.detect_statement_type()
statement_properties = bank_detector.statement_propertys

print(f'{bank} - {statement_type}')

bbva - credit


In [4]:
from table_boundary_detector import TransactionTableBoundaryDetector

boundary_detector = TransactionTableBoundaryDetector(extracted_words, statement_properties)

start_idx = boundary_detector.start_idx
end_idx = boundary_detector.end_idx

df_table = boundary_detector.get_filtered_table_words()

print(f'{start_idx} - {end_idx}')
df_table

394 - 1678


,page,text,x0,top,x1,bottom
0,2,Tarjeta,227.788956,281.966303,258.908956,291.966303
1,2,Titular,261.688956,281.966303,289.468956,291.966303
2,2,4772,292.248956,281.966303,314.488956,291.966303
3,2,1430,317.268956,281.966303,339.508956,291.966303
4,2,2212,342.288956,281.966303,364.528956,291.966303
...,...,...,...,...,...,...
1279,5,$21.60,300.799451,472.001747,328.519451,482.001747
1280,5,TOTAL,11.338621,489.757653,39.008621,499.757653
1281,5,IMPORTES:,41.128621,489.757653,86.038621,499.757653
1282,5,$,485.505081,489.757653,490.635081,499.757653


In [5]:
from row_segmenter import TransactionRowSegmenter

row_segmenter = TransactionRowSegmenter(df_table, statement_properties)

header_row = row_segmenter.get_header_row()
grouped_rows = row_segmenter.group_rows()

print(header_row)
grouped_rows

{'columns': ['FECHA AUTORIZACION', 'FECHA APLICACION', 'CONCEPTO', 'R.F.C.', 'REFERENCIA', 'IMPORTE CARGOS', 'IMPORTE ABONOS'], 'x0': [34.290365699999995, 105.80862669999999, 216.91163669999997, 385.08360669999996, 418.46208069999994, 492.14656969999993, 550.7827897], 'x1': [81.17036569999999, 145.9886267, 264.0816367, 407.34360669999995, 469.7420807, 528.8915696999999, 588.1527896999999]}


,row_group,text,words,top,bottom,page
0,0,Tarjeta Titular 4772 1430 2212 3620,"[(Tarjeta, 227.78895570000003, 258.90895570000...",281.966303,291.966303,2
1,1,FECHA FECHA CONCEPTO R.F.C. REFERENCIA IMPORTE...,"[(FECHA, 34.290365699999995, 62.52036569999999...",297.462445,319.462445,2
2,2,17/02/24 19/02/24 BMOVIL.PAGO TDC ******0518 $...,"[(17/02/24, 29.585359699999913, 67.22535969999...",325.060870,335.060870,2
3,3,Iva :$ 0.00 Interes: $ 0.00 Comisiones:$ 0.00 ...,"[(Iva, 15.252428700000024, 28.602428700000026)...",338.707484,348.707484,2
4,4,17/02/24 19/02/24 CABIN BAR ******0768 $ 466.0...,"[(17/02/24, 29.585357700000024, 67.22535770000...",361.882681,383.969295,2
...,...,...,...,...,...,...
94,94,Movimientos Efectuados Tarjeta Adicional 4772 ...,"[(Movimientos, 11.338633400000163, 67.47863340...",387.853085,397.853085,5
95,95,FECHA FECHA TIPO DE CONCEPTO R.F.C. REFERENCIA...,"[(FECHA, 30.694473400000177, 58.92447340000017...",406.135605,428.135605,5
96,96,27/02/24 28/02/24 CD TFL UNPAID FARE ******996...,"[(27/02/24, 25.989472800000158, 63.62947280000...",434.442692,456.489936,5
97,97,07/03/24 08/03/24 CD TFL UNPAID FARE ******594...,"[(07/03/24, 25.989466800000145, 63.62946680000...",459.954503,482.001747,5


In [6]:
from table_reconstructor import TransactionTableReconstructor

table_reconstructor = TransactionTableReconstructor(grouped_rows, header_row, statement_properties)
df_structured = table_reconstructor.get_structured_table()
df_structured

,FECHA AUTORIZACION,FECHA APLICACION,CONCEPTO,R.F.C.,REFERENCIA,IMPORTE CARGOS,IMPORTE ABONOS
0,17/02/24,19/02/24,BMOVIL.PAGO TDC,,******0518,,"21,823.56-"
1,17/02/24,19/02/24,CABIN BAR GBP $21.57 TIPO DE CAMBIO $21.61,,******0768,466.07,
2,17/02/24,19/02/24,SHELL CORNISH GATEWAY GBP $1.80 TIPO DE CAMBIO...,,******5567,38.89,
3,17/02/24,19/02/24,BRIOCHE DOREE 2961623 978 $5.40 TIPO DE CAMBIO...,,******6326,99.77,
4,17/02/24,19/02/24,L ADAGIO 978 $12.75 TIPO DE CAMBIO $18.48,,******0178,235.57,
...,...,...,...,...,...,...,...
68,13/03/24,14/03/24,Lycamobile UK GBP $5.00 TIPO DE CAMBIO $21.64,,******7922,108.18,
69,14/03/24,15/03/24,DOT APP DKK $24.00 TIPO DE CAMBIO $2.45,,******7694,58.81,
70,16/03/24,19/03/24,03 DE 03 VIVAAEROBUS,,,"1,385.28",
71,27/02/24,,CD TFL UNPAID FARE GBP $2.80 TIPO DE CAMBIO $2...,,******9960,$ 61.10,


In [7]:
from table_normalizer import TransactionTableNormalizer

normalizer = TransactionTableNormalizer(df_structured, extracted_words, statement_properties)

normalizer.normalize_table() # Final result

,Date,Description,Amount,Type
0,2024-02-17,BMOVIL.PAGO TDC,0.00,Saldo
1,2024-02-17,CABIN BAR GBP $21.57 TIPO DE CAMBIO $21.61,-466.07,Cargo
2,2024-02-17,SHELL CORNISH GATEWAY GBP $1.80 TIPO DE CAMBIO...,-38.89,Cargo
3,2024-02-17,BRIOCHE DOREE 2961623 978 $5.40 TIPO DE CAMBIO...,-99.77,Cargo
4,2024-02-17,L ADAGIO 978 $12.75 TIPO DE CAMBIO $18.48,-235.57,Cargo
...,...,...,...,...
68,2024-03-13,Lycamobile UK GBP $5.00 TIPO DE CAMBIO $21.64,-108.18,Cargo
69,2024-03-14,DOT APP DKK $24.00 TIPO DE CAMBIO $2.45,-58.81,Cargo
70,2024-03-16,03 DE 03 VIVAAEROBUS,-1385.28,Cargo
71,2024-02-27,CD TFL UNPAID FARE GBP $2.80 TIPO DE CAMBIO $2...,0.00,Saldo
